# 02 · Preparación reproducible

Se aplica la decisión de la tarjeta #3. La clase positiva corresponde a un puntaje de Matemáticas estrictamente inferior al percentil 20 de entrenamiento. Se conserva el corte completo y se reutiliza para todas las particiones. El criterio describe bajo rendimiento relativo dentro del prototipo.

El primer modelo usa curso, horas de estudio y participación extracurricular. La disponibilidad previa de estos campos debe comprobarse antes de un uso institucional. Los atributos sensibles quedan en los CSV de origen para una auditoría agregada posterior.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

carpeta = next((p for p in (Path("Datasets"), Path("../Datasets")) if p.is_dir()), None)
if carpeta is None:
    raise FileNotFoundError("Abre el notebook desde la raíz de EV3 o desde Notebooks.")

predictoras = ["Grade", "StudyHours", "Extracurricular"]
puntaje = "TestScore_Math"
tipos = {"Grade": "int64", "StudyHours": "float32",
         "Extracurricular": "int64", puntaje: "float64"}

def cargar_particion(nombre):
    datos = pd.read_csv(carpeta / f"{nombre}.csv", usecols=[*predictoras, puntaje],
                        dtype=tipos, float_precision="round_trip")
    validas = (
        datos["Grade"].between(9, 12)
        & np.isfinite(datos["StudyHours"]) & datos["StudyHours"].ge(0)
        & datos["Extracurricular"].isin([0, 1])
        & datos[puntaje].between(0, 100)
    )
    if not validas.all():
        raise ValueError(f"{nombre}: {(~validas).sum():,} filas fuera del esquema esperado")
    return datos

def preparar(datos, corte):
    X = datos[predictoras].astype({"Grade": "int8", "Extracurricular": "int8"})
    y = datos[puntaje].lt(corte).astype("int8").rename("riesgo")
    return X, y

In [2]:
train = cargar_particion("train")
corte_riesgo = float(train[puntaje].quantile(0.20, interpolation="linear"))
X_train, y_train = preparar(train, corte_riesgo)
del train

X_val, y_val = preparar(cargar_particion("validation"), corte_riesgo)
resumen = pd.DataFrame([
    {"partición": nombre, "filas": len(y), "clase positiva": int(y.sum()),
     "proporción positiva": round(float(y.mean()), 6)}
    for nombre, y in [("train", y_train), ("validation", y_val)]
])
print(f"Corte de riesgo: {corte_riesgo!r}")
print(resumen.to_string(index=False))
print("Variables predictoras:", ", ".join(X_train.columns))
assert list(X_train.columns) == list(X_val.columns) == predictoras
assert set(y_train.unique()) == set(y_val.unique()) == {0, 1}

Corte de riesgo: 66.57446853227711
 partición   filas  clase positiva  proporción positiva
     train 8000774         1600155             0.200000
validation  999229          200558             0.200713
Variables predictoras: Grade, StudyHours, Extracurricular


## Prueba reservada

Se verifica el contrato de entrada de prueba. La función `cargar_prueba()` se usará después de seleccionar el modelo y su umbral en validación. El umbral de decisión del modelo será una probabilidad. Es distinto del corte de puntaje que define la etiqueta.

In [3]:
filas_test = len(cargar_particion("test"))
print(f"Filas de prueba con esquema válido: {filas_test:,}")

def cargar_prueba():
    return preparar(cargar_particion("test"), corte_riesgo)

Filas de prueba con esquema válido: 999,997


## Paso a modelado

Las matrices y etiquetas se generan en memoria a partir de los CSV originales. Curso se puede tratar como ordinal en una primera comparación. Si un modelo requiere escalado u otra transformación aprendida, se ajustará con entrenamiento y se aplicará a validación y prueba. La comparación inicial usará un clasificador base, regresión logística y un modelo de árboles.